In [1]:
import numpy as np

from dataeval.config import set_seed
from dataeval.extractors._uncertainty import _prediction_uncertainty
from dataeval.shift import DriftDomainClassifier, DriftUnivariate

# Set random seed for reproducibility
set_seed(0, all_generators=True)
rng = np.random.default_rng(0)

In [2]:
num_samples = 1000
num_classes = 5

# Baseline: Model is confident (one class dominates)
prev_logits = rng.normal(loc=0, scale=1.0, size=(num_samples, num_classes))
prev_logits[:, 0] += 3.0  # Make class 0 the dominant, confident prediction
prev_probs = np.exp(prev_logits) / np.sum(np.exp(prev_logits), axis=1, keepdims=True)

# Drifted: Model is confused (probabilities are closer to uniform)
curr_logits = rng.normal(loc=0, scale=1.0, size=(num_samples, num_classes))
curr_probs = np.exp(curr_logits) / np.sum(np.exp(curr_logits), axis=1, keepdims=True)

In [3]:
prev_entropy = _prediction_uncertainty(prev_probs, preds_type="probs")
curr_entropy = _prediction_uncertainty(curr_probs, preds_type="probs")

# Use Kolmogorov-Smirnov (ks) test
detector = DriftUnivariate(method="ks").fit(prev_entropy)
result = detector.predict(curr_entropy)

print(f"Confidence Shift Detected: {result.drifted}")

Confidence Shift Detected: True


In [4]:
# TEST ASSERTION CELL ###
assert result.drifted

In [5]:
# Simulate a shift where class 2 becomes the dominant prediction instead of class 0
drifted_class_logits = rng.normal(loc=0, scale=1.0, size=(num_samples, num_classes))
drifted_class_logits[:, 2] += 3.0
curr_probs_marginal = np.exp(drifted_class_logits) / np.sum(np.exp(drifted_class_logits), axis=1, keepdims=True)

# Use Cramér-von Mises (cvm), which is highly sensitive to overall distributional differences
detector_marginal = DriftUnivariate(method="cvm").fit(prev_probs)
result_marginal = detector_marginal.predict(curr_probs_marginal)

print(f"Marginal Shift Detected: {result_marginal.drifted}")
drifted_classes = np.where(result_marginal.details["feature_drift"])[0]
print(f"Which classes drifted? {drifted_classes.tolist()}")

Marginal Shift Detected: True
Which classes drifted? [0, 2]


In [6]:
# TEST ASSERTION CELL ###
assert result_marginal.drifted
assert result_marginal.details["feature_drift"].any()

In [7]:
detector_multivariate = DriftDomainClassifier().fit(prev_probs)
result_multivariate = detector_multivariate.predict(curr_probs_marginal)

print(f"Multivariate Shift Detected: {result_multivariate.drifted} (AUROC: {result_multivariate.distance:.4f})")

Multivariate Shift Detected: True (AUROC: 0.9967)


In [8]:
# TEST ASSERTION CELL ###
assert result_multivariate.drifted

In [9]:
# Convert our probabilities to hard predictions (argmax)
prev_labels = np.argmax(prev_probs, axis=1)
curr_labels = np.argmax(curr_probs_marginal, axis=1)

# One-hot encode
prev_onehot = np.eye(num_classes)[prev_labels].astype(np.float32)
curr_onehot = np.eye(num_classes)[curr_labels].astype(np.float32)

detector_label = DriftUnivariate(method="cvm").fit(prev_onehot)
result_label = detector_label.predict(curr_onehot)

print(f"Label Shift Detected: {result_label.drifted}")

Label Shift Detected: True


In [10]:
# TEST ASSERTION CELL ###
assert result_label.drifted